In [ ]:
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from scipy.stats import pearsonr
from genetic_perturbation_playground.data.data_loader import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

adata = load_dataset("replogle_k562")

## Data preparation

Unlike scGen (which trains only on ctrl vs one target), CPA trains on **all perturbations simultaneously**.  
Each perturbation gets its own learned embedding; the encoder is trained adversarially to *not* encode perturbation identity in the basal state.  
We hold out the same RPL3 test split as the scGen notebook for a fair comparison.

In [ ]:
target_gene = "RPL3"
rng = np.random.default_rng(42)

def to_dense(X):
    return np.asarray(X.todense()) if sp.issparse(X) else np.asarray(X)

# Same RPL3 / ctrl splits as scGen for a fair comparison
ctrl_idx = np.where(adata.obs["perturbation"] == "control")[0]
rpl3_idx = np.where(adata.obs["perturbation"] == target_gene)[0]

ctrl_train_idx, ctrl_test_idx = train_test_split(ctrl_idx, test_size=0.2, random_state=42)
rpl3_train_idx, rpl3_test_idx = train_test_split(rpl3_idx, test_size=0.2, random_state=42)

# CPA trains on ALL perturbations — subsample to keep training tractable
MAX_PER_PERT = 50   # cells per perturbation
MAX_CTRL     = 500  # control cells (many more available; cap to balance)

train_indices = list(rng.choice(ctrl_train_idx, size=min(len(ctrl_train_idx), MAX_CTRL), replace=False))
train_indices += list(rpl3_train_idx)

for pert in adata.obs["perturbation"].cat.categories:
    if pert in ("control", target_gene):
        continue
    idx = np.where(adata.obs["perturbation"] == pert)[0]
    train_indices += list(rng.choice(idx, size=min(len(idx), MAX_PER_PERT), replace=False))

train_indices = np.array(train_indices)

ctrl_test_adata = adata[ctrl_test_idx].copy()
rpl3_test_adata = adata[rpl3_test_idx].copy()

# Integer-encode perturbation labels (needed for the embedding layer)
pert_cats = list(adata.obs["perturbation"].cat.categories)
pert_to_idx = {p: i for i, p in enumerate(pert_cats)}
n_perts = len(pert_cats)

X_train = torch.from_numpy(to_dense(adata[train_indices].X)).float()
P_train = torch.tensor(
    [pert_to_idx[p] for p in adata.obs["perturbation"].iloc[train_indices]],
    dtype=torch.long,
)
n_genes = X_train.shape[1]

print(f"Train: {len(X_train):,} cells | {n_perts} perturbations | {n_genes} genes")
print(f"Test RPL3: {len(rpl3_test_adata):,} | Test ctrl: {len(ctrl_test_adata):,}")

## CPA model

Three components (following Lotfollahi et al. 2023):

1. **VAE encoder** → `z_basal`: basal cell state, adversarially trained to contain *no* perturbation information  
2. **Perturbation embedding** → `z_pert`: one learnable vector per perturbation  
3. **Decoder**: reconstructs expression from `z_basal + z_pert`  

The **adversary** (a separate classifier on `z_basal`) is trained jointly:  
- Adversary minimises its own cross-entropy (tries to predict perturbation from `z_basal`)  
- Encoder maximises the adversary's cross-entropy (tries to remove perturbation signal from `z_basal`)  

At prediction time: encode a control cell → add the RPL3 embedding → decode.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, n_in, n_hidden, n_latent):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden), nn.LayerNorm(n_hidden), nn.ReLU(),
            nn.Linear(n_hidden, n_hidden), nn.LayerNorm(n_hidden), nn.ReLU(),
        )
        self.mu     = nn.Linear(n_hidden, n_latent)
        self.logvar = nn.Linear(n_hidden, n_latent)

    def forward(self, x):
        h = self.net(x)
        mu, lv = self.mu(h), self.logvar(h)
        z = mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
        return z, mu, lv


class CPAModel(nn.Module):
    def __init__(self, n_genes, n_perts, latent_dim=128, hidden_dim=512):
        super().__init__()
        self.encoder = Encoder(n_genes, hidden_dim, latent_dim)

        self.pert_emb = nn.Embedding(n_perts, latent_dim)
        nn.init.normal_(self.pert_emb.weight, std=0.01)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, n_genes),
        )
        # Adversary: classifies perturbation from z_basal — trained to succeed,
        # encoder trained to make it fail (disentanglement)
        self.adversary = nn.Sequential(
            nn.Linear(latent_dim, 128), nn.ReLU(),
            nn.Linear(128, n_perts),
        )

    def forward(self, x, pert_idx):
        z_basal, mu, lv = self.encoder(x)
        z_pert = self.pert_emb(pert_idx)
        x_hat  = self.decoder(z_basal + z_pert)
        return x_hat, mu, lv, z_basal

    @torch.no_grad()
    def predict(self, x_ctrl, pert_idx):
        _, mu, _ = self.encoder(x_ctrl)   # use mean at test time (no sampling)
        return self.decoder(mu + self.pert_emb(pert_idx))


model = CPAModel(n_genes, n_perts).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
BATCH_SIZE = 512
MAX_EPOCHS = 200
LR         = 3e-4
BETA_KL    = 0.01   # KL regularisation weight
ALPHA_ADV  = 0.5    # adversarial weight: how hard to penalise encoding perturbation in z_basal
PATIENCE   = 20

loader = DataLoader(
    TensorDataset(X_train, P_train),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"),
)

opt_main = torch.optim.Adam(
    list(model.encoder.parameters()) +
    list(model.decoder.parameters()) +
    list(model.pert_emb.parameters()),
    lr=LR,
)
opt_adv = torch.optim.Adam(model.adversary.parameters(), lr=LR)

best_loss, patience_ctr, best_state = float("inf"), 0, None

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    tot_recon = tot_kl = tot_adv = 0.0

    for X_b, P_b in loader:
        X_b, P_b = X_b.to(device), P_b.to(device)

        x_hat, mu, lv, z_basal = model(X_b, P_b)

        recon = F.mse_loss(x_hat, X_b)
        kl    = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1).mean()
        adv   = F.cross_entropy(model.adversary(z_basal), P_b)

        # Encoder/decoder: reconstruct well AND fool the adversary
        opt_main.zero_grad()
        (recon + BETA_KL * kl - ALPHA_ADV * adv).backward()
        opt_main.step()

        # Adversary: learn to predict perturbation from z_basal (detach so encoder isn't updated again)
        opt_adv.zero_grad()
        F.cross_entropy(model.adversary(z_basal.detach()), P_b).backward()
        opt_adv.step()

        tot_recon += recon.item()
        tot_kl    += kl.item()
        tot_adv   += adv.item()

    n = len(loader)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | recon={tot_recon/n:.4f}  kl={tot_kl/n:.4f}  adv={tot_adv/n:.4f}")

    if tot_recon < best_loss:
        best_loss, patience_ctr = tot_recon, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(best_state)
print(f"\nDone. Best recon loss: {best_loss/len(loader):.4f}")

In [ ]:
model.eval()

rpl3_pert_idx = pert_to_idx[target_gene]

# Counterfactual prediction:
# Take held-out control cells, encode to basal state, add RPL3 embedding, decode.
X_ctrl = torch.from_numpy(to_dense(ctrl_test_adata.X)).float().to(device)
pert_rpl3 = torch.full((len(ctrl_test_adata),), rpl3_pert_idx, dtype=torch.long, device=device)

X_pred = model.predict(X_ctrl, pert_rpl3).cpu().numpy()

mean_pred   = X_pred.mean(axis=0)
mean_actual = to_dense(rpl3_test_adata.X).mean(axis=0)
mean_ctrl   = to_dense(ctrl_test_adata.X).mean(axis=0)

r, _ = pearsonr(mean_pred - mean_ctrl, mean_actual - mean_ctrl)

print(f"Evaluation — {target_gene}")
print(f"  R² on differential (CPA):   {r**2:.4f}")
print(f"  R² on differential (scGen): 0.0987")